# 06 — Dose-Response Setup

Selects top hits from single-point screening and generates simulated 8-point
dose-response data for IC50 calculation.

**Requires:** `data/results/hits_75pct.csv`

In [1]:
import pandas as pd
import numpy as np

hits = pd.read_csv('data/results/hits_75pct.csv')
print(f'Total hits >75%: {len(hits)}')
hits.head()

Total hits >75%: 208


,Plate,Row,Column,Well,WellType,RFU,Pct_Activity
0,Plate 1,E,11,E11,Experimental,30089,99.54
1,Plate 1,G,13,G13,Experimental,29926,98.99
2,Plate 1,B,8,B08,Experimental,29818,98.63
3,Plate 1,K,9,K09,Experimental,29123,96.31
4,Plate 1,L,3,L03,Experimental,28059,92.76


## Select top 20 hits for dose-response

In a real screen, cherry-picked compounds would be reformatted into dose-response plates.
Here we select the top 20 by % activity across all plates.

In [2]:
top_hits = hits.nlargest(20, 'Pct_Activity').reset_index(drop=True)
top_hits['Compound_ID'] = [f'Cmpd_{i+1:02d}' for i in range(len(top_hits))]
print('Top 20 hits selected for dose-response:')
print(top_hits[['Compound_ID','Plate','Well','Pct_Activity']].to_string(index=False))

Top 20 hits selected for dose-response:
Compound_ID   Plate Well  Pct_Activity
    Cmpd_01 Plate 3  B21         99.90
    Cmpd_02 Plate 5  F04         99.89
    Cmpd_03 Plate 2  M09         99.79
    Cmpd_04 Plate 1  E11         99.54
    Cmpd_05 Plate 1  G13         98.99
    Cmpd_06 Plate 1  B08         98.63
    Cmpd_07 Plate 2  O05         98.29
    Cmpd_08 Plate 5  H17         98.27
    Cmpd_09 Plate 4  L03         98.20
    Cmpd_10 Plate 2  P14         98.19
    Cmpd_11 Plate 2  B15         97.84
    Cmpd_12 Plate 5  P19         97.74
    Cmpd_13 Plate 4  L09         97.40
    Cmpd_14 Plate 3  D21         97.18
    Cmpd_15 Plate 2  J06         97.14
    Cmpd_16 Plate 4  B19         97.09
    Cmpd_17 Plate 4  F13         96.99
    Cmpd_18 Plate 2  E13         96.53
    Cmpd_19 Plate 2  G12         96.44
    Cmpd_20 Plate 3  N09         96.32


## Generate 8-point dose-response data

Concentration series: 100, 30, 10, 3, 1, 0.3, 0.1, 0.03 µM (3-fold dilutions)

Each compound is assigned a random IC50 between 0.1–20 µM.
Simulated using the 4PL model with added noise.

In [3]:
def four_param_logistic(x, A, B, C, D):
    """4PL equation — ascending form (% inhibition increases with concentration)."""
    return A + (D - A) / (1.0 + (C / x) ** B)

concentrations = [100, 30, 10, 3, 1, 0.3, 0.1, 0.03]  # µM

np.random.seed(42)
records = []

for _, row in top_hits.iterrows():
    ic50_true = np.random.uniform(0.1, 20)     # random true IC50
    hill      = np.random.uniform(0.8, 2.0)    # Hill slope
    bottom    = np.random.uniform(0, 8)        # baseline noise
    top       = np.random.uniform(88, 100)     # max inhibition

    for conc in concentrations:
        inh = four_param_logistic(conc, bottom, hill, ic50_true, top)
        inh += np.random.normal(0, 2)          # assay noise
        inh = round(np.clip(inh, 0, 105), 2)
        records.append({
            'Compound_ID':      row['Compound_ID'],
            'Source_Plate':     row['Plate'],
            'Source_Well':      row['Well'],
            'SP_Pct_Activity':  row['Pct_Activity'],
            'Concentration_uM': conc,
            'Inhibition_pct':   inh,
            'True_IC50_uM':     round(ic50_true, 3)  # kept for reference
        })

dr_df = pd.DataFrame(records)
print(f'Dose-response records: {len(dr_df)}')
dr_df.head(16)

Dose-response records: 160


,Compound_ID,Source_Plate,Source_Well,SP_Pct_Activity,Concentration_uM,Inhibition_pct,True_IC50_uM
0,Cmpd_01,Plate 3,B21,99.90,100.00,94.13,7.553
1,Cmpd_01,Plate 3,B21,99.90,30.00,88.97,7.553
2,Cmpd_01,Plate 3,B21,99.90,10.00,65.55,7.553
3,Cmpd_01,Plate 3,B21,99.90,3.00,20.15,7.553
4,Cmpd_01,Plate 3,B21,99.90,1.00,6.65,7.553
5,Cmpd_01,Plate 3,B21,99.90,0.30,7.11,7.553
6,Cmpd_01,Plate 3,B21,99.90,0.10,4.95,7.553
7,Cmpd_01,Plate 3,B21,99.90,0.03,4.93,7.553
8,Cmpd_02,Plate 5,F04,99.89,100.00,87.86,6.154
9,Cmpd_02,Plate 5,F04,99.89,30.00,83.84,6.154


## Save dose-response data

In [4]:
dr_df.to_csv('data/results/dose_response_data.csv', index=False)
print(f'Saved: data/results/dose_response_data.csv')
print(f'Compounds: {dr_df["Compound_ID"].nunique()}, Concentrations: {len(concentrations)} points each')

Saved: data/results/dose_response_data.csv
Compounds: 20, Concentrations: 8 points each
